# ⚙️ EidCloud Job Engine - Interactive Quickstart

Welcome to the **EidCloud Job Engine** interactive tour! This notebook demonstrates:
1. **Environment Setup & Job Dispatching** (Priorities, Delays, Rate Limiting, Chaining)
2. **Parallel Worker Processing** (Consuming jobs, handling retries with exponential backoff)
3. **Dead Letter Queue (DLQ) Inspection & Replay** (Auditing failed jobs and replaying them back to active queues)

In [ ]:
# Cell 1: Environment Setup & Dispatching Jobs
import os
import subprocess
import json

# Ensure PHP is installed
print("Checking PHP version:")
subprocess.run(["php", "-v"], check=True)

# Create sample job classes and dispatch jobs
php_script = """
<?php
require_once __DIR__ . '/../src/Contracts/JobInterface.php';
require_once __DIR__ . '/../src/Contracts/QueueDriverInterface.php';
require_once __DIR__ . '/../src/Models/JobPayload.php';
require_once __DIR__ . '/../src/Drivers/SqliteQueueDriver.php';
require_once __DIR__ . '/../src/Worker/WorkerOptions.php';
require_once __DIR__ . '/../src/Worker/Worker.php';
require_once __DIR__ . '/../src/JobQueue.php';

class EmailNotificationJob implements \EidCloud\JobEngine\Contracts\JobInterface {
    public function handle(array $payload): mixed {
        echo "[EmailJob] Sending email to: " . ($payload['to'] ?? 'unknown') . "\n";
        return true;
    }
}

class FragileApiJob implements \EidCloud\JobEngine\Contracts\JobInterface {
    public function handle(array $payload): mixed {
        echo "[FragileJob] Attempting external API call...\n";
        throw new \RuntimeException("External Gateway Timeout (HTTP 504)");
    }
}

$dbFile = sys_get_temp_dir() . '/quickstart_queue.sqlite';
@unlink($dbFile);
$engine = new \EidCloud\JobEngine\JobQueue('sqlite', ['path' => $dbFile]);

// 1. High priority email job
$id1 = $engine->dispatch(EmailNotificationJob::class, ['to' => 'ceo@eidcloud.com'], queue: 'critical', priority: 100);

// 2. Default email job
$id2 = $engine->dispatch(EmailNotificationJob::class, ['to' => 'user@domain.com'], queue: 'default', priority: 50);

// 3. Fragile job designed to fail into DLQ (1 retry max)
$id3 = $engine->dispatch(FragileApiJob::class, ['endpoint' => '/v1/charge'], queue: 'default', maxAttempts: 1);

echo "Dispatched Jobs Successfully!\n";
echo "- Critical Job: {$id1}\n";
echo "- Default Job:  {$id2}\n";
echo "- Fragile Job:  {$id3}\n";
echo "\nInitial Stats: " . json_encode($engine->stats(), JSON_PRETTY_PRINT) . "\n";
""";

with open("quickstart_step1.php", "w") as f:
    f.write(php_script)

result = subprocess.run(["php", "quickstart_step1.php"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

In [ ]:
# Cell 2: Parallel Background Worker Processing
step2_script = """
<?php
require_once __DIR__ . '/../src/Contracts/JobInterface.php';
require_once __DIR__ . '/../src/Contracts/QueueDriverInterface.php';
require_once __DIR__ . '/../src/Models/JobPayload.php';
require_once __DIR__ . '/../src/Drivers/SqliteQueueDriver.php';
require_once __DIR__ . '/../src/Worker/WorkerOptions.php';
require_once __DIR__ . '/../src/Worker/Worker.php';
require_once __DIR__ . '/../src/JobQueue.php';
require_once __DIR__ . '/quickstart_step1.php'; // Load job classes

$dbFile = sys_get_temp_dir() . '/quickstart_queue.sqlite';
$engine = new \EidCloud\JobEngine\JobQueue('sqlite', ['path' => $dbFile]);

echo "Starting worker consuming queues: [critical, default]...\n";
$options = new \EidCloud\JobEngine\Worker\WorkerOptions(stopWhenEmpty: true);
$worker = $engine->worker(['critical', 'default'], $options);
$worker->work();

echo "\nWorker Execution Complete!\n";
echo "- Processed Jobs: " . $worker->getProcessedCount() . "\n";
echo "- Failed Jobs:    " . $worker->getFailedCount() . "\n";
echo "\nUpdated Stats: " . json_encode($engine->stats(), JSON_PRETTY_PRINT) . "\n";
""";

with open("quickstart_step2.php", "w") as f:
    f.write(step2_script)

result = subprocess.run(["php", "quickstart_step2.php"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

In [ ]:
# Cell 3: Dead Letter Queue (DLQ) Inspection & Replay
step3_script = """
<?php
require_once __DIR__ . '/../src/Contracts/JobInterface.php';
require_once __DIR__ . '/../src/Contracts/QueueDriverInterface.php';
require_once __DIR__ . '/../src/Models/JobPayload.php';
require_once __DIR__ . '/../src/Drivers/SqliteQueueDriver.php';
require_once __DIR__ . '/../src/JobQueue.php';

$dbFile = sys_get_temp_dir() . '/quickstart_queue.sqlite';
$engine = new \EidCloud\JobEngine\JobQueue('sqlite', ['path' => $dbFile]);

$dlq = $engine->dlq();
echo "=== Inspecting Dead Letter Queue ===\n";
echo "Total in DLQ: " . count($dlq) . "\n\n";

foreach ($dlq as $item) {
    echo "[DLQ Entry]\n";
    echo "  Job ID:      " . $item['id'] . "\n";
    echo "  Class:       " . $item['job_class'] . "\n";
    echo "  Attempts:    " . $item['attempts'] . "\n";
    echo "  Error:       " . $item['error_message'] . "\n";
    
    echo "  -> Replaying job back into active queue...\n";
    $replayed = $engine->replay($item['id']);
    echo "  -> Replay status: " . ($replayed ? "SUCCESS" : "FAILED") . "\n\n";
}

echo "Stats After Replay:\n" . json_encode($engine->stats(), JSON_PRETTY_PRINT) . "\n";
""";

with open("quickstart_step3.php", "w") as f:
    f.write(step3_script)

result = subprocess.run(["php", "quickstart_step3.php"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

# Cleanup temporary scratch scripts
for s in ["quickstart_step1.php", "quickstart_step2.php", "quickstart_step3.php"]:
    if os.path.exists(s):
        os.remove(s)